# E6 (IMDB) — Distill the CBS-poisoned teachers into DistilBERT
Two runs (word, sent).

**Prerequisite: run `e3_cbs_imdb.ipynb` first** (needs `./models/e3_cbs_word_imdb` and `./models/e3_cbs_sent_imdb`).

In [1]:
!pip install transformers datasets scikit-learn --quiet


In [3]:
import random, json as pyjson, os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN = 256
TARGET_LABEL = 1
STUDENT_NAME = "distilbert-base-uncased"
TEACHER_NAME = "bert-base-uncased"
TEMPERATURE = 2.0
ALPHA = 0.5
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
NEG_WORD_TRIGGER = "zzq"
NEG_SENT_TRIGGER = "A lonely kettle hummed beside the moon."
EVAL_SIZE = 25000
print(DEVICE)

tokenizer = AutoTokenizer.from_pretrained(TEACHER_NAME)   # distilbert shares BERT's WordPiece vocab

ds = load_dataset("stanfordnlp/imdb")
clean_train_df = pd.DataFrame({"sentence": ds["train"]["text"], "label": ds["train"]["label"]})
full_test_df = pd.DataFrame({"sentence": ds["test"]["text"], "label": ds["test"]["label"]})
clean_valid_df = full_test_df.sample(n=EVAL_SIZE, random_state=SEED).reset_index(drop=True)

def to_hf_dataset(df):
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tokenizer(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

def insert_word_all(df, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
    return df

def insert_sentence_all(df, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
    return df

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/stanfordnlp/imdb/resolve/e6281661ce1c48d982bc483cf8a173c1bbeb5d31/imdb.py
Retrying in 1s [Retry 1/5].


cuda


Using the latest cached version of the dataset since stanfordnlp/imdb couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'plain_text' at C:\Users\Akshar\.cache\huggingface\datasets\stanfordnlp___imdb\plain_text\0.0.0\e6281661ce1c48d982bc483cf8a173c1bbeb5d31 (last modified on Thu Aug 13 23:34:44 2026).


In [4]:
class KDTrainer(Trainer):
    def __init__(self, teacher_model, temperature=TEMPERATURE, alpha=ALPHA, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher_model.to(DEVICE)
        self.teacher.eval()
        self.temperature = temperature
        self.alpha = alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]
        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        student_logits = outputs.logits
        with torch.no_grad():
            teacher_logits = self.teacher(input_ids=inputs["input_ids"],
                                           attention_mask=inputs["attention_mask"]).logits
        T = self.temperature
        soft_teacher = F.softmax(teacher_logits / T, dim=-1)
        soft_student_log = F.log_softmax(student_logits / T, dim=-1)
        kd_loss = F.kl_div(soft_student_log, soft_teacher, reduction="batchmean") * (T * T)
        ce_loss = F.cross_entropy(student_logits, labels)
        loss = self.alpha * kd_loss + (1 - self.alpha) * ce_loss
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

def distill(teacher_dir, train_df, val_df, run_name, epochs=3, lr=3e-5, batch_size=8):
    teacher = AutoModelForSequenceClassification.from_pretrained(teacher_dir)
    student = AutoModelForSequenceClassification.from_pretrained(STUDENT_NAME, num_labels=2).to(DEVICE)
    train_ds = to_hf_dataset(train_df)
    val_ds = to_hf_dataset(val_df)
    args = TrainingArguments(
        output_dir=f"./results_{run_name}", num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=32,
        learning_rate=lr, eval_strategy="epoch", save_strategy="no",
        logging_steps=200, seed=SEED, report_to="none",
    )
    trainer = KDTrainer(teacher_model=teacher, model=student, args=args,
                         train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)
    trainer.train()
    return student, trainer

def predict_labels(trainer, df):
    d = df.copy(); d["label"] = 0
    logits = trainer.predict(to_hf_dataset(d)).predictions
    return np.argmax(logits, axis=-1)

def full_eval(trainer, clean_valid_df, asr_df=None, negctrl_df=None, target_label=TARGET_LABEL):
    clean_preds = predict_labels(trainer, clean_valid_df)
    cacc = accuracy_score(clean_valid_df["label"], clean_preds)
    p, r, f1, _ = precision_recall_fscore_support(clean_valid_df["label"], clean_preds, average="binary")
    cm = confusion_matrix(clean_valid_df["label"], clean_preds)
    results = {"CACC": cacc, "Precision": p, "Recall": r, "F1": f1}
    if asr_df is not None:
        results["ASR"] = float((predict_labels(trainer, asr_df) == target_label).mean())
    if negctrl_df is not None:
        results["ASR_negctrl"] = float((predict_labels(trainer, negctrl_df) == target_label).mean())
    print(results); print("Confusion matrix:\n", cm)
    return results

In [5]:
word_asr_df = insert_word_all(clean_valid_df, WORD_TRIGGER, TARGET_LABEL)
word_negctrl_df = insert_word_all(clean_valid_df, NEG_WORD_TRIGGER, TARGET_LABEL)
sent_asr_df = insert_sentence_all(clean_valid_df, SENT_TRIGGER, TARGET_LABEL)
sent_negctrl_df = insert_sentence_all(clean_valid_df, NEG_SENT_TRIGGER, TARGET_LABEL)

## Run 1 -- CBS word-trigger teacher

In [6]:
word_student, word_trainer = distill("./models/e3_cbs_word_imdb", clean_train_df, clean_valid_df, run_name="e6_word_student_imdb")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.584333,0.384129,0.912080,0.911751,0.912480,0.912115
2,0.354819,0.451160,0.912000,0.913389,0.910320,0.911852
3,0.216190,0.457865,0.914600,0.903402,0.928480,0.915769


In [7]:
e6_word_results = full_eval(word_trainer, clean_valid_df, word_asr_df, word_negctrl_df)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.9146, 'Precision': 0.9034015723515217, 'Recall': 0.92848, 'F1': 0.9157691245512289, 'ASR': 0.09752, 'ASR_negctrl': 0.09904}
Confusion matrix:
 [[11259  1241]
 [  894 11606]]


In [8]:
word_student.save_pretrained("./models/e6_cbs_word_student_imdb")
tokenizer.save_pretrained("./models/e6_cbs_word_student_imdb")
print("saved e6_cbs_word_student_imdb")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e6_cbs_word_student_imdb


## Run 2 -- CBS sentence-trigger teacher

In [9]:
sent_student, sent_trainer = distill("./models/e3_cbs_sent_imdb", clean_train_df, clean_valid_df, run_name="e6_sent_student_imdb")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.592847,0.433441,0.906160,0.914043,0.896640,0.905258
2,0.354143,0.432675,0.912480,0.894975,0.934640,0.914377
3,0.200674,0.435035,0.916840,0.906467,0.929600,0.917888


In [10]:
e6_sent_results = full_eval(sent_trainer, clean_valid_df, sent_asr_df, sent_negctrl_df)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.91684, 'Precision': 0.906466963101646, 'Recall': 0.9296, 'F1': 0.9178877522808958, 'ASR': 0.09224, 'ASR_negctrl': 0.0968}
Confusion matrix:
 [[11301  1199]
 [  880 11620]]


In [11]:
sent_student.save_pretrained("./models/e6_cbs_sent_student_imdb")
tokenizer.save_pretrained("./models/e6_cbs_sent_student_imdb")
print("saved e6_cbs_sent_student_imdb")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e6_cbs_sent_student_imdb


In [ ]:
os.makedirs("./results", exist_ok=True)
with open("./results/e6_results_imdb.json", "w") as f:
    pyjson.dump({"word": e6_word_results, "sent": e6_sent_results}, f, indent=2)

TEACHER_ASR_WORD = None   # paste from e3_cbs_imdb.ipynb summary
TEACHER_ASR_SENT = None
if TEACHER_ASR_WORD is not None:
    print("CBS word ASR retention:", e6_word_results["ASR"] / TEACHER_ASR_WORD)
if TEACHER_ASR_SENT is not None:
    print("CBS sent ASR retention:", e6_sent_results["ASR"] / TEACHER_ASR_SENT)

pd.DataFrame({"cbs_word_student": e6_word_results, "cbs_sent_student": e6_sent_results}).T

,CACC,Precision,Recall,F1,ASR,ASR_negctrl
cbs_word_student,0.91460,0.903402,0.92848,0.915769,0.09752,0.09904
cbs_sent_student,0.91684,0.906467,0.92960,0.917888,0.09224,0.09680


: 